In [13]:
import xml.etree.ElementTree as ET
import numpy as np
import scipy as sp
from scipy.io import whosmat, loadmat

In [3]:
from pathlib import Path

chemin = Path("/mnt/hubel-data-149/Rat012/Rat012_2025-12-15")


In [4]:
for f in sorted(chemin.iterdir()):
    print(f.name)

BeforeCheckSameClu
In
InfraSlowRhythm
IniClu
Rat012_2025-12-15.cat.evt
Rat012_2025-12-15.cat.evt.old
Rat012_2025-12-15.clu.1
Rat012_2025-12-15.clu.10
Rat012_2025-12-15.clu.11
Rat012_2025-12-15.clu.12
Rat012_2025-12-15.clu.12.03.05.2026.14.46
Rat012_2025-12-15.clu.2
Rat012_2025-12-15.clu.3
Rat012_2025-12-15.clu.4
Rat012_2025-12-15.clu.5
Rat012_2025-12-15.clu.6
Rat012_2025-12-15.clu.7
Rat012_2025-12-15.clu.8
Rat012_2025-12-15.clu.9
Rat012_2025-12-15.dat
Rat012_2025-12-15.deltaWaves.events.mat
Rat012_2025-12-15.drowsiness
Rat012_2025-12-15.fet.1
Rat012_2025-12-15.fet.10
Rat012_2025-12-15.fet.11
Rat012_2025-12-15.fet.12
Rat012_2025-12-15.fet.2
Rat012_2025-12-15.fet.3
Rat012_2025-12-15.fet.4
Rat012_2025-12-15.fet.5
Rat012_2025-12-15.fet.6
Rat012_2025-12-15.fet.7
Rat012_2025-12-15.fet.8
Rat012_2025-12-15.fet.9
Rat012_2025-12-15.fil
Rat012_2025-12-15.klg.1
Rat012_2025-12-15.klg.10
Rat012_2025-12-15.klg.11
Rat012_2025-12-15.klg.12
Rat012_2025-12-15.klg.2
Rat012_2025-12-15.klg.3
Rat012_2025-12-

In [5]:
HPC_groups =[7, 8, 9]
MPFC_groups =[10, 11, 12]  

hpc_files =[]
mpfc_files =[]

for g in HPC_groups:
    hpc_files.append({
        "group": g,
        "clu" : next(chemin.glob(f"*.clu.{g}")),
        "res" : next(chemin.glob(f"*.res.{g}"))
    })

for g in MPFC_groups:
    mpfc_files.append({
        "group": g,
        "clu" : next(chemin.glob(f"*.clu.{g}")),
        "res" : next(chemin.glob(f"*.res.{g}"))
    })

print("HPC:")
for x in hpc_files:
    print("HPC groupe", x["group"])
    print("clu:", x["clu"])  
    print("res:", x["res"]) 

print("mPFC:")
for x in mpfc_files:
    print("mPFC groupe", x["group"])
    print("clu:", x["clu"])  
    print("res:", x["res"]) 

HPC:
HPC groupe 7
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.7
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.7
HPC groupe 8
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.8
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.8
HPC groupe 9
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.9
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.9
mPFC:
mPFC groupe 10
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.10
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.10
mPFC groupe 11
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.11
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.11
mPFC groupe 12
clu: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.clu.12
res: /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.res.12


In [6]:
xml_file = next(chemin.glob("*.xml"))
tree = ET.parse(xml_file)
root = tree.getroot()

sampling_rate = None

for elem in root.iter():
    if elem.tag.lower().endswith("samplingrate"):
        sampling_rate = float(elem.text)
        break

print("fichier xml:", xml_file.name)
print("freq d'ech :", sampling_rate, "Hz")

fichier xml: Rat012_2025-12-15.xml
freq d'ech : 20000.0 Hz


In [7]:
def load_clu_res(clu_path, res_path):

    clu_raw = np.loadtxt(clu_path, dtype=np.int64)
    res = np.loadtxt(res_path, dtype=np.int64)

    clu = clu_raw[1:]

    if len(clu) != len(res):
        raise ValueError(
            f"Longueurs différentes : clu={len(clu)}, res={len(res)}"
        ) 

    return clu, res

In [8]:
FE = 20_000

def load_units(files, fs):
    units ={}

    for x in files:
        group = x["group"]

        clu, res = load_clu_res(x["clu"], x["res"])
        spike_times = res / FE

        for cluster in np.unique(clu):
            units[(group, cluster)]= spike_times[clu == cluster]

    return units

hpc_units = load_units(hpc_files, FE)
mpfc_units = load_units(mpfc_files, FE)     

In [9]:
delta_files =[
    f for f in chemin.iterdir()
    if "delta" in f.name.lower()
] 

for f in delta_files:
    print(f.name)

Rat012_2025-12-15.deltaWaves.events.mat


In [10]:
delta_file= delta_files[0] 

print("Fichier :", delta_file)
print("Taille :", delta_file.stat().st_size / 1024, "KB")

Fichier : /mnt/hubel-data-149/Rat012/Rat012_2025-12-15/Rat012_2025-12-15.deltaWaves.events.mat
Taille : 103.7431640625 KB


In [11]:
with open(delta_file, "rb") as f:
    header = f.read(100)

print(header)

b'MATLAB 5.0 MAT-file, Platform: GLNXA64, Created on: Tue Mar 31 15:44:19 2026                        '


In [12]:
for x in whosmat(delta_file):
    print(x)

('deltaWaves', (1, 1), 'struct')


In [19]:
mat = loadmat(delta_file, simplify_cells=True)

print(mat.keys())

deltawaves = mat["deltaWaves"]

print(type(deltawaves)) 
print(deltawaves.keys())

dict_keys(['__header__', '__version__', '__globals__', 'deltaWaves'])
<class 'dict'>
dict_keys(['timestamps', 'peaks', 'peakNormedPower', 'detectorName', 'troughValue', 'badIntervals'])


In [20]:
for key in ["timestamps", "peaks"]:
    x = np.asarray(deltawaves[key])

    print("\n", key)
    print("shape:", x.shape)
    print("dtype:", x.dtype)
    print("premières valeurs:")
    print(x[:10]) 


 timestamps
shape: (3338, 2)
dtype: float64
premières valeurs:
[[507.7656 508.0152]
 [789.5    789.86  ]
 [802.044  802.316 ]
 [842.6312 842.9016]
 [846.8552 847.1416]
 [861.3624 861.7264]
 [872.0072 872.3392]
 [874.3592 874.6456]
 [876.1632 876.3248]
 [878.7104 879.0536]]

 peaks
shape: (3338,)
dtype: float64
premières valeurs:
[507.8808 789.7328 802.1736 842.7584 846.9912 861.5904 872.1384 874.5032
 876.164  878.8808]


In [21]:
delta_peaks = np.asarray(deltawaves["peaks"], dtype=float)
print("nombre de delta waves:", len(delta_peaks))
print(delta_peaks[:10])

nombre de delta waves: 3338
[507.8808 789.7328 802.1736 842.7584 846.9912 861.5904 872.1384 874.5032
 876.164  878.8808]


In [32]:
def build_hpc_matrix(hpc_units, delta_peaks, window=0.200):
    unit_ids = list(hpc_units.keys())
    X = np.zeros((len(delta_peaks), len(unit_ids)), dtype=int)

    for j, unit_id in enumerate(unit_ids):
        spikes = np.asarray(hpc_units[unit_id])

        left = np.searchsorted(spikes, delta_peaks - window, side = "left")
        right = np.searchsorted(spikes, delta_peaks, side = "right")

        X[:, j] = right - left

    return X, unit_ids 

In [33]:
X_hpc, hpc_unit_ids = build_hpc_matrix (hpc_units, delta_peaks)

print("Shape X_hpc:", X_hpc.shape)
print("Unités HPC:", hpc_unit_ids)

Shape X_hpc: (3338, 61)
Unités HPC: [(7, np.int64(3)), (7, np.int64(7)), (7, np.int64(9)), (7, np.int64(10)), (7, np.int64(13)), (7, np.int64(17)), (7, np.int64(20)), (7, np.int64(25)), (7, np.int64(31)), (7, np.int64(33)), (7, np.int64(35)), (7, np.int64(38)), (7, np.int64(41)), (7, np.int64(42)), (7, np.int64(44)), (7, np.int64(50)), (7, np.int64(51)), (7, np.int64(52)), (7, np.int64(55)), (7, np.int64(58)), (7, np.int64(67)), (7, np.int64(68)), (7, np.int64(71)), (7, np.int64(73)), (7, np.int64(76)), (7, np.int64(80)), (7, np.int64(84)), (7, np.int64(89)), (7, np.int64(96)), (7, np.int64(101)), (7, np.int64(103)), (7, np.int64(104)), (7, np.int64(105)), (7, np.int64(106)), (8, np.int64(5)), (8, np.int64(6)), (8, np.int64(13)), (8, np.int64(23)), (8, np.int64(26)), (8, np.int64(28)), (8, np.int64(29)), (8, np.int64(32)), (8, np.int64(33)), (8, np.int64(45)), (8, np.int64(46)), (9, np.int64(3)), (9, np.int64(7)), (9, np.int64(8)), (9, np.int64(10)), (9, np.int64(11)), (9, np.int64(15)

In [34]:
hpc_units = {
    k: value 
    for k, value in hpc_units.items()
    if k[1] > 1 
} 

mpfc_units = {
    k: value 
    for k, value in mpfc_units.items()
    if k[1] > 1 
} 

In [35]:
print("HPC:", len(hpc_units), "unités")
print("mPFC:", len(mpfc_units), "unités")

print(list(hpc_units.keys()))
print(list(mpfc_units.keys()))

HPC: 61 unités
mPFC: 22 unités
[(7, np.int64(3)), (7, np.int64(7)), (7, np.int64(9)), (7, np.int64(10)), (7, np.int64(13)), (7, np.int64(17)), (7, np.int64(20)), (7, np.int64(25)), (7, np.int64(31)), (7, np.int64(33)), (7, np.int64(35)), (7, np.int64(38)), (7, np.int64(41)), (7, np.int64(42)), (7, np.int64(44)), (7, np.int64(50)), (7, np.int64(51)), (7, np.int64(52)), (7, np.int64(55)), (7, np.int64(58)), (7, np.int64(67)), (7, np.int64(68)), (7, np.int64(71)), (7, np.int64(73)), (7, np.int64(76)), (7, np.int64(80)), (7, np.int64(84)), (7, np.int64(89)), (7, np.int64(96)), (7, np.int64(101)), (7, np.int64(103)), (7, np.int64(104)), (7, np.int64(105)), (7, np.int64(106)), (8, np.int64(5)), (8, np.int64(6)), (8, np.int64(13)), (8, np.int64(23)), (8, np.int64(26)), (8, np.int64(28)), (8, np.int64(29)), (8, np.int64(32)), (8, np.int64(33)), (8, np.int64(45)), (8, np.int64(46)), (9, np.int64(3)), (9, np.int64(7)), (9, np.int64(8)), (9, np.int64(10)), (9, np.int64(11)), (9, np.int64(15)), (9

In [36]:
X_hpc, hpc_unit_ids = build_hpc_matrix (hpc_units, delta_peaks)

print("Dimensions X_hpc:", X_hpc.shape)

Dimensions X_hpc: (3338, 61)


In [37]:
def build_mpfc_targets(mpfc_units, delta_peaks, half_window=0.015):
    targets = {} 
    
    for unit_id, spikes in mpfc_units.items():
        spikes = np.asarray(spikes)

        left = np.searchsorted(spikes, delta_peaks - half_window, side = "left")
        right = np.searchsorted(spikes, delta_peaks + half_window, side = "right")

        targets[unit_id] = (right > left).astype(int) 

    return targets

In [38]:
y_mpfc = build_mpfc_targets (mpfc_units, delta_peaks)

print("Nombre de neurones mPFC:", len(y_mpfc))

Nombre de neurones mPFC: 22
